In [ ]:
# Make this notebook work from either fine-tuning/ or fine-tuning/live-demo/
# (idempotent: re-running is safe)
import os
from pathlib import Path
_here = Path.cwd()
if _here.name in ('pre-demo', 'live-demo'):
    os.chdir(_here.parent)
print('cwd:', Path.cwd())


# Lab 05 · Conversation Memory — stop making her repeat herself (LIVE)

> **LIVE DEMO LAB.** Resources are suffixed `-live` so this run never overwrites the pre-demo lab.

A returning member hates repeating herself, but a stateless agent forgets her the moment she hangs up. This lab gives the agent memory three ways: a sliding window for the current call, a rolling summary for long conversations, and a persistent per-member profile across calls. When she calls back the next day, the agent already knows her preferred pharmacy and active meds. *She never has to start over.*

---
## Step 1 — Config & client

*Load `.env` and build the `AzureOpenAI` client we'll reuse for every turn.*


In [ ]:
# Standard config + client. Same .env all labs share.
import os, json, time
from pathlib import Path
from dotenv import load_dotenv
from openai import AzureOpenAI
from azure.identity import DefaultAzureCredential, get_bearer_token_provider

load_dotenv()
AZURE_OPENAI_ENDPOINT    = os.environ['AZURE_OPENAI_ENDPOINT']
AZURE_OPENAI_API_VERSION = os.environ.get('AZURE_OPENAI_API_VERSION', '2025-04-01-preview')
BASE_DEPLOYMENT          = os.environ.get('BASE_DEPLOYMENT', 'gpt-4o-mini')

_cred = DefaultAzureCredential()
_tp   = get_bearer_token_provider(_cred, 'https://cognitiveservices.azure.com/.default')

client = AzureOpenAI(
    azure_endpoint = AZURE_OPENAI_ENDPOINT,
    azure_ad_token_provider = _tp,
    api_version    = AZURE_OPENAI_API_VERSION,
)
print('Client OK. Deployment:', BASE_DEPLOYMENT)


---
## Step 2 — Sliding-window memory

*Keep only the last N turns verbatim and replay them on every call. Cheap, stateless, fine for short conversations.*


In [ ]:
SYS = 'You are a Sutter Health voice agent. Be warm and concise.'

class WindowMemory:
    def __init__(self, max_turns=6):
        self.msgs = []
        self.max_turns = max_turns
    def add(self, role, content):
        self.msgs.append({'role': role, 'content': content})
        # keep only last max_turns user+assistant pairs
        self.msgs = self.msgs[-self.max_turns*2:]
    def prompt(self):
        return [{'role':'system','content':SYS}] + self.msgs

def chat(memory, user_msg):
    memory.add('user', user_msg)
    r = client.chat.completions.create(
        model=BASE_DEPLOYMENT,
        messages=memory.prompt(),
        temperature=0.3, max_tokens=180,
    )
    reply = r.choices[0].message.content
    memory.add('assistant', reply)
    return reply

m = WindowMemory(max_turns=4)
print('A>', chat(m, "Hi, my name is Maria. My member id is MEM-099."))
print()
print('A>', chat(m, "I need to refill my metformin."))
print()
print('A>', chat(m, "What's my name?"))   # should remember


---
## Step 3 — When the window drops important context

*Shrink the window so the early turns get dropped — watch the agent forget the member id it was just told.*


In [ ]:
m = WindowMemory(max_turns=2)   # only last 2 turns kept
print('A>', chat(m, "Hi, my name is Maria. My id is MEM-099."))
print()
print('A>', chat(m, "I take metformin every morning."))
print()
print('A>', chat(m, "I have a pollen allergy."))
print()
print('A>', chat(m, "Remind me - what is my member id?"))   # likely forgotten


---
## Step 4 — Rolling-summary memory

*When the window overflows, summarize the **dropped** turns into one short note and keep that note forever. Constant cost, infinite memory.*


In [ ]:
class SummaryMemory:
    def __init__(self, window=4):
        self.summary = ''
        self.recent = []
        self.window = window
    def add(self, role, content):
        self.recent.append({'role': role, 'content': content})
        # When we exceed window, summarize and drop
        if len(self.recent) > self.window*2:
            to_sum = self.recent[:2]   # drop oldest pair
            self.recent = self.recent[2:]
            s = client.chat.completions.create(
                model=BASE_DEPLOYMENT,
                messages=[
                    {'role':'system','content':'Summarize the following exchange in one short paragraph. Preserve names, ids, and any clinical or policy facts.'},
                    {'role':'user','content':json.dumps(to_sum)},
                ],
                temperature=0.0, max_tokens=120,
            )
            new_sum = s.choices[0].message.content.strip()
            self.summary = (self.summary + ' ' + new_sum).strip() if self.summary else new_sum
    def prompt(self):
        sys = SYS + ('\n\nWHAT YOU ALREADY KNOW: ' + self.summary if self.summary else '')
        return [{'role':'system','content':sys}] + self.recent

m = SummaryMemory(window=2)
print('A>', chat(m, "Hi, I'm Maria, MEM-099. I take metformin daily."))
print()
print('A>', chat(m, "I prefer CVS on Geary for pickup."))
print()
print('A>', chat(m, "I also have a peanut allergy."))
print()
print('A>', chat(m, "Where do I usually pick up my prescriptions?"))
print('\n--- summary captured ---')
print(m.summary)


---
## Step 5 — Persistent member memory (the Sutter killer feature)

*Save what we learned about the member to disk. On the next call, load the profile into the system prompt so the agent walks in already knowing them.*


In [ ]:
PROFILE_PATH = Path('data/member_profiles.json')
PROFILE_PATH.parent.mkdir(parents=True, exist_ok=True)
if not PROFILE_PATH.exists():
    PROFILE_PATH.write_text('{}', encoding='utf-8')

def load_profile(member_id):
    return json.loads(PROFILE_PATH.read_text(encoding='utf-8')).get(member_id, {})

def save_profile(member_id, profile):
    db = json.loads(PROFILE_PATH.read_text(encoding='utf-8'))
    db[member_id] = profile
    PROFILE_PATH.write_text(json.dumps(db, indent=2), encoding='utf-8')

# Today's call - we learn things
save_profile('MEM-099', {
    'name': 'Maria Rodriguez',
    'preferred_pharmacy': 'CVS - 2645 Geary Blvd, San Francisco',
    'active_meds': ['metformin 500mg', 'lisinopril 10mg'],
    'allergies': ['peanut'],
    'communication_pref': 'text',
})
print('Saved profile for MEM-099.')

# Tomorrow's call - she calls back
profile = load_profile('MEM-099')
mem_sys = (
    SYS +
    f"\n\nMEMBER ON THIS CALL:\n{json.dumps(profile, indent=2)}\n"
    "Use this to skip questions you already know the answer to."
)
r = client.chat.completions.create(
    model=BASE_DEPLOYMENT,
    messages=[
        {'role':'system','content':mem_sys},
        {'role':'user','content':"Hi, it's Maria again. Can you send my refill to my usual spot?"},
    ],
    temperature=0.3, max_tokens=200,
)
print('\nA>', r.choices[0].message.content)


---
## Step 6 — Takeaway

*Wrap-up: when each of the three strategies is the right pick.*
